In [1]:
!pip install -q "transformers[torch]" Pillow scikit-learn seaborn tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Gerät: {device}")

model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id).to(device)
model.eval()
print(f"Modell geladen: {model_id}")

In [ ]:
categories = ["indoor", "outdoor"]

# Mixed Prompts (empfohlen nach Liao et al. 2025)
category_texts = [
    "a photo taken inside a room showing walls, ceiling, furniture, artificial lighting or floor",
    "a photo taken outdoors showing open spaces such as nature, sky, streets, buildings, parks or landscapes"
]

# Alternativ: Simple Prompts
# category_texts = [
#     "This is an indoor photo.",
#     "This is an outdoor photo."
# ]

# Alternativ: Detailed Prompts
# category_texts = [
#     "a photo taken inside a building showing rooms, offices, shops, restaurants or other enclosed spaces with visible walls, ceilings, artificial lighting or furniture",
#     "a photo taken outside in the open air showing natural environments, urban streets, parks, forests, mountains, water bodies or any scene under natural light or open sky"
# ]

text_inputs = processor(text=category_texts, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    text_out = model.get_text_features(**text_inputs)
    text_features = text_out if isinstance(text_out, torch.Tensor) else text_out.pooler_output
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print("Kategorien:", categories)
print("Text-Embeddings berechnet:", text_features.shape)

In [ ]:
image_folder = "/content/drive/MyDrive/flickr_data_subset/inside_outside_photos"

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_paths = []

for fname in sorted(os.listdir(image_folder)):
    if fname.startswith("augmented_"):
        continue
    ext = os.path.splitext(fname)[1].lower()
    if ext not in VALID_EXTENSIONS:
        continue
    fpath = os.path.join(image_folder, fname)
    if os.path.isfile(fpath):
        image_paths.append(fpath)

print(f"Bilder gefunden: {len(image_paths)}")

In [ ]:
def safe_open(path):
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        print(f"  ⚠ Bild übersprungen ({os.path.basename(path)}): {e}")
        return None

BATCH_SIZE = 32

results     = []
confidences = []
valid_paths = []

for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Batches"):
    batch_paths = image_paths[i : i + BATCH_SIZE]

    loaded = [(p, safe_open(p)) for p in batch_paths]
    loaded = [(p, img) for p, img in loaded if img is not None]

    if not loaded:
        continue

    paths_ok, imgs_ok = zip(*loaded)

    img_inputs = processor(
        images=list(imgs_ok),
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        image_out = model.get_image_features(**img_inputs)
        image_features = image_out if isinstance(image_out, torch.Tensor) else image_out.pooler_output
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

    similarities = image_features @ text_features.T

    batch_preds = similarities.argmax(dim=1).cpu().numpy()
    batch_conf  = similarities.max(dim=1).values.cpu().float().numpy()

    results.extend([categories[idx] for idx in batch_preds])
    confidences.extend(batch_conf)
    valid_paths.extend(paths_ok)

print(f"\nKlassifiziert: {len(results)} Bilder")
print("\nVorhersage-Verteilung:")
for cat in categories:
    print(f"  {cat}: {results.count(cat)}")

In [ ]:
output_base = "/content/drive/MyDrive/flickr_data_subset/flickr_photos_sorted"

sorted_dirs = {cat: os.path.join(output_base, cat) for cat in categories}
for d in sorted_dirs.values():
    os.makedirs(d, exist_ok=True)

for path, pred in zip(valid_paths, results):
    shutil.copy(path, sorted_dirs[pred])

print("Bilder sortiert:")
for cat in categories:
    count = results.count(cat)
    print(f"  {cat}: {count} Bilder → {sorted_dirs[cat]}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(confidences, bins=30, alpha=0.7, color="steelblue")
plt.xlabel("Cosine Similarity")
plt.ylabel("Anzahl Bilder")
plt.title("Confidence-Verteilung aller Vorhersagen")
plt.tight_layout()
plt.show()

In [ ]:
output_csv = "/content/drive/MyDrive/flickr_data_subset/flickr_photos_sorted/clip_labels_confidence.csv"

df_results = pd.DataFrame({
    "file_name":  [os.path.basename(p) for p in valid_paths],
    "pred_label": results,
    "confidence": confidences
})

df_results.to_csv(output_csv, index=False)
print(f"Labels gespeichert: {output_csv}")
print()
print("Verteilung:")
print(df_results["pred_label"].value_counts())
print(f"\nGesamt: {len(df_results)} Bilder")

In [ ]:
df_results["confidence"] = confidences
df_unsicher = df_results.nsmallest(10, "confidence")
print(df_unsicher)

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

output_dir = "/content/drive/MyDrive/flickr_data_subset/flickr_photos_sorted/evaluation"
os.makedirs(output_dir, exist_ok=True)

label_csv   = "/content/drive/MyDrive/flickr_data_subset/flickr_photos_sorted/photo_labels_indoor_outdoor_17042026.csv"
df_labels   = pd.read_csv(label_csv)
labels_dict = dict(zip(df_labels["file_name"], df_labels["Main_focus"]))

df_results["true_label"] = df_results["file_name"].map(labels_dict)
df_eval = df_results.dropna(subset=["true_label"])
print(f"Evaluierbare Bilder: {len(df_eval)}")

true     = df_eval["true_label"].values
pred     = df_eval["pred_label"].values
accuracy = accuracy_score(true, pred)
avg_conf = float(np.mean(df_eval["confidence"]))

print(f"\nAccuracy:              {accuracy:.4f}")
print(f"Ø Confidence (Cosine): {avg_conf:.4f}")
print()
report = classification_report(true, pred,
                                target_names=sorted(categories),
                                output_dict=True)
print(classification_report(true, pred, target_names=sorted(categories)))

pd.DataFrame(report).transpose().to_csv(
    os.path.join(output_dir, "clip_metrics.csv"))

le = LabelEncoder()
le.classes_ = np.array(sorted(categories))
cm = confusion_matrix(le.transform(true), le.transform(pred))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix – ViT-B/32 (Accuracy: {accuracy:.2%})")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "clip_confusion_matrix.png"),
            dpi=150, bbox_inches='tight')
plt.show()

correct      = (df_eval["true_label"] == df_eval["pred_label"]).tolist()
conf_correct = [c for c, ok in zip(df_eval["confidence"], correct) if ok]
conf_wrong   = [c for c, ok in zip(df_eval["confidence"], correct) if not ok]

plt.figure(figsize=(8, 4))
plt.hist(conf_correct, bins=30, alpha=0.7,
         label=f"Correct (n={len(conf_correct)})", color="steelblue")
plt.hist(conf_wrong,   bins=30, alpha=0.7,
         label=f"Wrong   (n={len(conf_wrong)})",   color="tomato")
plt.xlabel("Cosine Similarity")
plt.ylabel("Anzahl Bilder")
plt.title("Confidence-Verteilung: korrekte vs. falsche Vorhersagen")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "clip_confidence_distribution.png"),
            dpi=150, bbox_inches='tight')
plt.show()

df_uncertain = df_eval.nsmallest(10, "confidence")[
    ["file_name", "true_label", "pred_label", "confidence"]]
print("\nUnsicherste 10 Bilder:")
print(df_uncertain)
df_uncertain.to_csv(os.path.join(output_dir, "clip_most_uncertain.csv"), index=False)

print(f"\n── Alle Dateien gespeichert in: {output_dir} ──")